Symetryczny system szyfrowania to taki, w którym klucz szyfrujący pozwala zarówno szyfrować dane, jak również odszyfrowywać je Podstawową wadą systemów symetrycznych jest ścisła konieczność ochrony klucza. Z tego powodu można je było stosować tylko w ograniczonych grupach użytkowników.

Algorytm RSA jest niesymetryczny, którego zasadniczą cechą są dwa klucze: publiczny do kodowania informacji oraz prywatny do jej odczytywania. Klucz publiczny umożliwia jedynie zaszyfrowanie danych i w żaden sposób nie ułatwia ich odczytania, nie musi być chroniony

Drugi klucz (prywatny, przechowywany pod nadzorem) służy do odczytywania informacji zakodowanych przy pomocy pierwszego klucza. Klucz ten nie jest udostępniany publicznie.

Algorytm RSA składa się z trzech podstawowych kroków:

Generacja klucza publicznego i tajnego. Klucz publiczny jest przekazywany wszystkim zainteresowanym i umożliwia zaszyfrowanie danych. Klucz tajny umożliwia rozszyfrowanie danych zakodowanych kluczem publicznym. Jest przechowywany on bezpiecznie bez możliwości udostępniania.

Użytkownik po otrzymaniu klucza publicznego koduje za jego pomocą swoje dane i przesyła je w postaci szyfru RSA do  adresata dysponującego kluczem tajnym

Klucz publiczny nie musi być chroniony, ponieważ nie umożliwia on rozszyfrowania informacji – proces szyfrowania nie jest odwracalny przy pomocy tego klucza.
Adresat po otrzymaniu zaszyfrowanej wiadomości rozszyfrowuje ją za pomocą klucza tajnego.




In [20]:
import struct
import zlib

from Crypto.Util.number import getPrime
import math

p i q to Liczby pierwsze
phi to funkcja Eulera, n jest modułem

In [21]:
def nwd(a,b):
    while b != 0:
        a,b = b, a % b
    return a
def find_e(phi_n):
    e = 3
    while nwd(e ,phi_n) != 1:
        e+= 2
    return e
def generate_rsa_keys(bits=1024):

    p = getPrime(bits // 2)
    q = getPrime(bits // 2)

    n = p * q
    phi_n = (p-1) * (q-1)

    #e = 65537
    e = find_e(phi_n)

    # rozszerzony algorytm Euklidesa jako funkcja pow() w Pythonie pozwala na obliczenie odwrotności modularnej. Operacaja potęgi modulo
    #Odwrotnośc modulo - liczba x, która po pomnożeniu przez zadaną liczbę "a" daje reśztę z dzielenia przez n równą 1
    d = pow(e, -1, phi_n)

    public_key = (e,n)
    private_key = (d,n)

    return public_key, private_key

In [22]:
def encrypt_rsa(message, public_key):
    e, n = public_key

    c = pow(message,e,n)
    return c

In [23]:
def decrypt_rsa(ciphertext, private_key):
    d, n = private_key

    m = pow(ciphertext,d,n)
    return m

In [24]:
def get_combined_idat(filepath):
    combined_idat_data = bytearray()

    with open(filepath, 'rb') as file:
        signature = file.read(8)

        while True:
            length_bytes = file.read(4)
            if not length_bytes:
                break

            chunk_length = struct.unpack(">I", length_bytes)[0]
            chunk_type = file.read(4).decode('ascii')
            chunk_data = file.read(chunk_length)
            chunk_crc = file.read(4)

            if chunk_type == 'IDAT':
                combined_idat_data.extend(chunk_data)
            elif chunk_type == 'IEND':
                break
        return bytes(combined_idat_data)

- input musi być zawsze mniejszy niż rozmiar klucza, stąd 100 bajtów dla 1024 bitowego klucza
- output musi być zawsze równy rozmiarowi klucza, stąd 128 bajtów dla 1024 bitowego klucza

## Padding

Wartość bajtu dopełnienia jest równa liczbie brakujących bajtów.
- Jeśli do dopełnienia brakuje np. 50 bajtów, dodaje się na końcu 50 bajtów, z których każdy ma wartośc 50
- Jeśli nic nie brakuje do dopełniania, to zgodnie ze standardem i tak dodajemy nowy blok, w którym każdy ma wartość rozmiaru pliku wejściowego)

In [25]:
def encrypt_bytes_ecb(data_bytes, pub_key, in_bloc_size=100, out_block_size=128):
    padding_len = in_bloc_size - (len(data_bytes) % in_bloc_size)
    if padding_len == 0:
        padding_len = in_bloc_size
    data_bytes = data_bytes + bytes([padding_len] * padding_len)

    encrypted_data = bytearray()
    for i in range(0, len(data_bytes), in_bloc_size):
        block = data_bytes[i:i+in_bloc_size]
        m = int.from_bytes(block, byteorder='big')
        c = encrypt_rsa(m, pub_key)
        encrypted_data.extend(c.to_bytes(out_block_size, byteorder='big'))
    return bytes(encrypted_data)

In [26]:
import binascii
def create_png_chunk(chunk_type, chunk_data):
    type_bytes = chunk_type.encode('ascii')
    crc = binascii.crc32(type_bytes + chunk_data)

    length_bytes = struct.pack('>I', len(chunk_data))
    crc_bytes = struct.pack('>I', crc)

    return length_bytes + type_bytes + chunk_data + crc_bytes

In [27]:
import struct

def save_encrypted_png(input_filepath, output_filepath, new_idat_chunk):
    with open(input_filepath, 'rb') as infile, open(output_filepath, 'wb') as outfile:
        signature = infile.read(8)
        outfile.write(signature)

        idat_written = False

        while True:
            length_bytes = infile.read(4)
            if not length_bytes:
                break

            chunk_length = struct.unpack('>I', length_bytes)[0]
            chunk_type_bytes = infile.read(4)
            chunk_type = chunk_type_bytes.decode('ascii')
            chunk_data = infile.read(chunk_length)
            chunk_crc = infile.read(4)

            original_chunk = length_bytes + chunk_type_bytes + chunk_data + chunk_crc

            if chunk_type == 'IDAT':
                if not idat_written:
                    outfile.write(new_idat_chunk)
                    idat_written = True
            else:
                outfile.write(original_chunk)

            if chunk_type == 'IEND':
                break

In [28]:
def process_png_method_a(idat_compressed_bytes, pub_key):
    raw = zlib.decompress(idat_compressed_bytes)
    encrypted = encrypt_bytes_ecb(raw, pub_key)
    recompressed_data = zlib.compress(encrypted)

    return create_png_chunk('IDAT', recompressed_data)

In [29]:
def process_png_method_b(idat_compressed_bytes, pub_key):
    encrypted = encrypt_bytes_ecb(idat_compressed_bytes, pub_key)

    return create_png_chunk('IDAT', encrypted)

In [30]:
def decrypt_bytes_ecb(encrypted_data_bytes, priv_key, in_block_size=128, out_block_size=100):
    decrypted_data = bytearray()
    for i in range(0, len(encrypted_data_bytes), in_block_size):
        block = encrypted_data_bytes[i:i+in_block_size]
        c = int.from_bytes(block, byteorder='big')
        m = decrypt_rsa(c, priv_key)
        decrypted_data.extend(m.to_bytes(out_block_size, byteorder='big'))

    padding_len = decrypted_data[-1]
    decrypted_data = decrypted_data[:-padding_len]
    return bytes(decrypted_data)

In [31]:
def process_png_decrypt_method_a(encrypted_idat_bytes, priv_key):
   encrypted = zlib.decompress(encrypted_idat_bytes)
   raw = decrypt_bytes_ecb(encrypted, priv_key)
   recompressed_data = zlib.compress(raw)
   return create_png_chunk('IDAT',recompressed_data)

In [32]:
def process_png_decrypt_method_b(encrypted_idat_bytes, priv_key):
    decrypted_compressed_data = decrypt_bytes_ecb(encrypted_idat_bytes, priv_key)

    return create_png_chunk('IDAT', decrypted_compressed_data)

Ponieważ pierwszy blok danych nie ma poprzedniego bloku, w trybie CBC stosuje się wektor inicjalizujący (IV), który jest losową wartością o długości bloku. Wektor inicjalizujący powinien być unikalny dla każdej operacji szyfrowania, aby zapewnić bezpieczeństwo danych.

In [33]:
import os

def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

def encrypt_bytes_cbc(data_bytes, pub_key, iv, in_block_size=100, out_block_size=128):
    padding_len = in_block_size - (len(data_bytes) % in_block_size)
    if padding_len == 0:
        padding_len = in_block_size
    data_bytes = data_bytes + bytes([padding_len] * padding_len)

    encrypted_data = bytearray()
    prev = iv

    for i in range(0, len(data_bytes), in_block_size):
        block = data_bytes[i:i+in_block_size]

        x = xor_bytes(block, prev)
        m = int.from_bytes(x, byteorder='big')


        c = encrypt_rsa(m, pub_key)
        c_bytes = c.to_bytes(out_block_size, byteorder='big')

        encrypted_data.extend(c_bytes)

        prev = c_bytes[:in_block_size]

    return bytes(encrypted_data)

def decrypt_bytes_cbc(encrypted_data_bytes, priv_key, iv, in_block_size=128, out_block_size=100):
    decrypted_data = bytearray()
    prev = iv

    for i in range(0, len(encrypted_data_bytes), in_block_size):
        block = encrypted_data_bytes[i:i+in_block_size]
        c = int.from_bytes(block, byteorder='big')

        m = decrypt_rsa(c, priv_key)
        x = m.to_bytes(out_block_size, byteorder='big')

        original = xor_bytes(x, prev)
        decrypted_data.extend(original)

        prev = block[:out_block_size]

    padding_len = decrypted_data[-1]

    decrypted_data = decrypted_data[:-padding_len]

    return bytes(decrypted_data)

In [34]:
def process_png_cbc_method_a(idat_compressed_bytes, pub_key, iv):
    raw = zlib.decompress(idat_compressed_bytes)
    encrypted = encrypt_bytes_cbc(raw, pub_key, iv)
    recompressed_data = zlib.compress(encrypted)
    return create_png_chunk('IDAT', recompressed_data)

def process_png_decrypt_cbc_method_a(encrypted_idat_bytes, priv_key, iv):
    encrypted = zlib.decompress(encrypted_idat_bytes)
    raw = decrypt_bytes_cbc(encrypted, priv_key, iv)
    recompressed_data = zlib.compress(raw)
    return create_png_chunk('IDAT', recompressed_data)

def process_png_cbc_method_b(idat_compressed_bytes, pub_key, iv):
    encrypted = encrypt_bytes_cbc(idat_compressed_bytes, pub_key, iv)
    return create_png_chunk('IDAT', encrypted)

def process_png_decrypt_cbc_method_b(encrypted_idat_bytes, priv_key, iv):
    decrypted = decrypt_bytes_cbc(encrypted_idat_bytes, priv_key, iv)
    return create_png_chunk('IDAT', decrypted)

In [35]:
filepath = "../flower.png"
pub_key, priv_key = generate_rsa_keys(bits=1024)

original_combined_idat = get_combined_idat(filepath)

encrypted_idat_a = process_png_method_a(original_combined_idat, pub_key)
crypted_path_a = "cryptedapple_method_a.png"
save_encrypted_png(input_filepath=filepath, output_filepath=crypted_path_a, new_idat_chunk=encrypted_idat_a)

read_encrypted_idat_a = get_combined_idat(crypted_path_a)
decrypted_idat_a = process_png_decrypt_method_a(read_encrypted_idat_a, priv_key)
save_encrypted_png(crypted_path_a, "decrypted_apple_method_a.png", decrypted_idat_a)

encrypted_idat_b = process_png_method_b(original_combined_idat, pub_key)
crypted_path_b = "cryptedapple_method_b.png"
save_encrypted_png(input_filepath=filepath, output_filepath=crypted_path_b, new_idat_chunk=encrypted_idat_b)

read_encrypted_idat_b = get_combined_idat(crypted_path_b)
decrypted_idat_b = process_png_decrypt_method_b(read_encrypted_idat_b, priv_key)
save_encrypted_png(crypted_path_b, "decrypted_apple_method_b.png", decrypted_idat_b)

In [36]:
iv = os.urandom(100)

encrypted_idat_cbc_a = process_png_cbc_method_a(original_combined_idat, pub_key, iv)
crypted_path_cbc_a = "cryptedapple_cbc_method_a.png"
save_encrypted_png(filepath, crypted_path_cbc_a, encrypted_idat_cbc_a)

read_encrypted_idat_cbc_a = get_combined_idat(crypted_path_cbc_a)
decrypted_idat_cbc_a = process_png_decrypt_cbc_method_a(read_encrypted_idat_cbc_a, priv_key, iv)
save_encrypted_png(crypted_path_cbc_a, "decrypted_apple_cbc_method_a.png", decrypted_idat_cbc_a)

encrypted_idat_cbc_b = process_png_cbc_method_b(original_combined_idat, pub_key, iv)
crypted_path_cbc_b = "cryptedapple_cbc_method_b.png"
save_encrypted_png(filepath, crypted_path_cbc_b, encrypted_idat_cbc_b)

read_encrypted_idat_cbc_b = get_combined_idat(crypted_path_cbc_b)
decrypted_idat_cbc_b = process_png_decrypt_cbc_method_b(read_encrypted_idat_cbc_b, priv_key, iv)
save_encrypted_png(crypted_path_cbc_b, "decrypted_apple_cbc_method_b.png", decrypted_idat_cbc_b)

Ocena czytelności danych po zaszyfrowaniu

ECB (Metoda A):

Kwiatek zamienił się w szum, nadal można dostrzec jego położenie na środku kadru. Najciemniejszy punkt w centrum kwiatka stworzył bloki danych, które po zaszyfrowaniu wyróżniają się od reszty. ECB zawodzi w ukrywaniu korelacji między sąsiednimi pikselami.

CBC (Metoda A):

W trybie CBC zjawisko poziomych pasów wywołanych filtrami PNG całkowicie znika. Wprowadzenie wektora inicjalizującego ($IV$) i operacji XOR sprawia, że informacja o kolorze i filtrze z poprzedniego piksela bezpośrednio wpływa na wygląd kolejnego bloku. Nie ma śladu po strukturze geometrycznej oryginału. CBC nie pokazuje lokalnych zależności danych.

ECB i CBC (Metoda B):

Pliki te są zepsute, dlatego że szyfrując bezpośrednio skompresowany blok IDAT, zmieniana jest losowo struktura danych.

### Porównanie metod A i B

- metody nie są równoważne
- szyfrowanie skompresowanych danych niszczy nagłówki strumienia kompresji, co uniemożliwia poprawne odczytanie pliku PNG
- Szyfrowanie zdekompresowanych pikseli i próba ich ponownej kompresji udowadnia, że szyfrogramu nie da się skompresować, ponieważ ma on zbyt dużą entropię (nie ma w nim wzorców)


## Porównanie z gotową funkcją RSA

Szyfrujemy **tą samą parą kluczy** (te same `n`, `e`, `d`) gotową, biblioteczną funkcją RSA (`PKCS1_OAEP` z PyCryptodome) i porównujemy wynik z naszą implementacją.

In [42]:
from Crypto.Cipher import PKCS1_OAEP
from Crypto.PublicKey import RSA
import struct

def oaep_encrypt(data, oaep, in_block_size):
    result = bytearray(struct.pack('>I', len(data)))
    for i in range(0, len(data), in_block_size):
        result.extend(oaep.encrypt(data[i:i + in_block_size]))
    return bytes(result)

def oaep_decrypt(data, oaep, key_size_bytes):
    original_len = struct.unpack('>I', data[:4])[0]
    result = bytearray()
    for i in range(4, len(data), key_size_bytes):
        block = data[i:i + key_size_bytes]
        if len(block) == key_size_bytes:
            result.extend(oaep.decrypt(block))
    return bytes(result[:original_len])

def process_png_encrypt_lib_oaep(idat_bytes, pub_key_tuple, method='b'):
    e, n = pub_key_tuple
    oaep = PKCS1_OAEP.new(RSA.construct((n, e)))

    key_bytes = (n.bit_length() + 7) // 8
    in_block = key_bytes - 42

    data = zlib.decompress(idat_bytes) if method == 'a' else idat_bytes
    encrypted = oaep_encrypt(data, oaep, in_block)
    return create_png_chunk('IDAT', zlib.compress(encrypted) if method == 'a' else encrypted)

def process_png_decrypt_lib_oaep(encrypted_idat_bytes, priv_key_tuple, method='b'):
    e, n = pub_key
    d = priv_key_tuple[0]
    oaep = PKCS1_OAEP.new(RSA.construct((n, e, d)))

    key_bytes = (n.bit_length() + 7) // 8

    data = zlib.decompress(encrypted_idat_bytes) if method == 'a' else encrypted_idat_bytes
    decrypted = oaep_decrypt(data, oaep, key_bytes)
    return create_png_chunk('IDAT', zlib.compress(decrypted) if method == 'a' else decrypted)

In [43]:
# Metoda A
enc_a = process_png_encrypt_lib_oaep(original_combined_idat, pub_key, method='a')
save_encrypted_png(filepath, "crypted_oaep_method_a.png", enc_a)

read_enc_a = get_combined_idat("crypted_oaep_method_a.png")
dec_a = process_png_decrypt_lib_oaep(read_enc_a, priv_key, method='a')
save_encrypted_png("crypted_oaep_method_a.png", "decrypted_oaep_method_a.png", dec_a)

# Metoda B
enc_b = process_png_encrypt_lib_oaep(original_combined_idat, pub_key, method='b')
save_encrypted_png(filepath, "crypted_oaep_method_b.png", enc_b)

read_enc_b = get_combined_idat("crypted_oaep_method_b.png")
dec_b = process_png_decrypt_lib_oaep(read_enc_b, priv_key, method='b')
save_encrypted_png("crypted_oaep_method_b.png", "decrypted_oaep_method_b.png", dec_b)

In [38]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import PKCS1_OAEP

e, n = pub_key
d, _ = priv_key

lib_key = RSA.construct((n, e, d))
oaep = PKCS1_OAEP.new(lib_key)

sample = original_combined_idat[:50]

c_ours = encrypt_rsa(int.from_bytes(sample, 'big'), pub_key)
c_ours_bytes = c_ours.to_bytes(128, 'big')

c_lib_1 = oaep.encrypt(sample)
c_lib_2 = oaep.encrypt(sample)

print("Nasz RSA      (hex, pierwsze 32 B):", c_ours_bytes[:32].hex())
print("Biblioteka #1 (hex, pierwsze 32 B):", c_lib_1[:32].hex())
print("Biblioteka #2 (hex, pierwsze 32 B):", c_lib_2[:32].hex())
print()
print("Nasz RSA deterministyczny:", c_ours_bytes == encrypt_rsa(int.from_bytes(sample, 'big'), pub_key).to_bytes(128, 'big'))
print("Biblioteka OAEP losowa:", c_lib_1 != c_lib_2)
print("Nasz wynik rozny od biblioteki:", c_ours_bytes != c_lib_1)
print()
c_lib_raw = pow(int.from_bytes(sample, 'big'), lib_key.e, lib_key.n)
print("Nasz RSA == surowe m^e mod n z klucza biblioteki:", c_ours == c_lib_raw)
print("Biblioteka odszyfrowuje swoj szyfrogram:", oaep.decrypt(c_lib_1) == sample)

Nasz RSA      (hex, pierwsze 32 B): 0c245de3cca1c223c38a1748036a599adeadd1667ed839afda42fc13276767cc
Biblioteka #1 (hex, pierwsze 32 B): 6c7d6c78dfaefaa03370daffa3ddb43c07c8eabfb3a7ed7ce67b1202fd42ed1b
Biblioteka #2 (hex, pierwsze 32 B): 588b9afc02105fb3f01af64f33428480347701a9e4020f6c45e1ead6685704b4

Nasz RSA deterministyczny: True
Biblioteka OAEP losowa: True
Nasz wynik rozny od biblioteki: True

Nasz RSA == surowe m^e mod n z klucza biblioteki: True
Biblioteka odszyfrowuje swoj szyfrogram: True


Nasz RSA deterministyczny: True
lgorytm dla tych samych danych wejściowych zawsze wygeneruje identyczny ciąg bajtów

Biblioteka OAEP losowa: True
Wywołanie funkcji oaep.encrypt(sample) dwa razy z rzędu dla tych samych 50 bajtów dało dwa całkowicie różne wyniki

Nasz RSA == m^e mod n: True
Potwierdzenie poprawności matematycznej

Dlaczego wyniki gotowej funkcji bibliotecznej są inne? Funkcja dodatkowo:
- generuje losowy ciąg bajtów (tzw. seed) przy każdym uruchomieniu szyfrowania
- maskuje dane za pomocą funkcji skrótu (np. SHA-256) i operacji XOR
Surowe RSA może prowadzić do ujawnienia struktury pliku